# XGBoost Regression - Network Throughput Prediction

Target: `Data Throughput (Mbps)`

Features: `Locality`, `Latitude`, `Longitude`, `Signal Strength (dBm)`, `Latency (ms)`, `Network Type`

## 1. Clone repo (Colab only)


In [1]:
!git clone https://github.com/MohamedAAbdullah1/NTI-Final-Project.git
%cd NTI-Final-Project

Cloning into 'NTI-Final-Project'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 71 (delta 23), reused 38 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 3.11 MiB | 6.50 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/NTI-Final-Project


## 2. Imports

In [27]:
import os
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## 3. Load data

In [3]:
X_train = pd.read_csv("data/processed/X_train.csv")
X_test = pd.read_csv("data/processed/X_test.csv")
y_train = pd.read_csv("data/processed/y_train.csv").values.ravel()
y_test = pd.read_csv("data/processed/y_test.csv").values.ravel()

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test:", X_test.shape, "| y_test:", y_test.shape)
X_train.head()

X_train: (13463, 6) | y_train: (13463,)
X_test: (3366, 6) | y_test: (3366,)


,Locality,Latitude,Longitude,Signal Strength (dBm),Latency (ms),Network Type
0,Pataliputra,25.605539,85.285651,-87.611830,164.101054,4G
1,Kidwaipuri,25.491973,85.086669,-90.089704,22.851745,5G
2,Danapur,25.684972,84.989785,-90.175391,139.812003,4G
3,Kankarbagh,25.569868,85.138714,-93.384826,113.904981,4G
4,Phulwari Sharif,25.626811,85.136823,-95.888704,71.291449,4G


## 4. Build pipeline (encode categoricals + XGBoost)

In [39]:
categorical_cols = ["Locality", "Network Type"]
numeric_cols = ["Latitude", "Longitude", "Signal Strength (dBm)", "Latency (ms)"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough"
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("xgb", xgb.XGBRegressor(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.03,
        subsample=1.0,
        colsample_bytree=0.9,
        min_child_weight=7,
        gamma=0.3,
        random_state=42

    ))
])

## 5. Train

In [40]:
model.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Locality',
                                                   'Network Type'])])),
                ('xgb',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=None,
                              colsample_bytree=0.9, device=None,
                              early_stopping_rounds=None,
                              enable_categoric...
                              feature_types=None, feature_weights=None,
                              gamma=0.3, grow_policy=None, importance_type=None,
                              interaction_constraints=None, learning_rate=0.03,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=3, max_leaves=None, min_child_weight=7,
                              missing=nan, monotone_constraints=None,
                              multi_strategy=None, n_estimators=200,
                              n_jobs=None, num_parallel_tree=None, ...))])

## 6. Evaluate

In [41]:
y_pred = search.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R²:", r2)

RMSE: 13.324216306364585
MAE: 6.4593060350241345
R²: 0.7360798758445786


## 7. Save model

In [42]:
os.makedirs("models", exist_ok=True)
model.named_steps["xgb"].save_model("models/xgb_model.json")
print("Saved to models/xgb_model.json")

Saved to models/xgb_model.json


## 8. Push to GitHub (Colab)

In [43]:
!git config --global user.email "your_email@example.com"
!git config --global user.name "MohamedAAbdullah1"

!git add notebooks/03_xgboost_model.ipynb models/xgb_model.json
!git commit -m "Add XGBoost regression model for throughput prediction"
!git push https://<YOUR_TOKEN>@github.com/MohamedAAbdullah1/NTI-Final-Project.git main

[main 1543094] Add XGBoost regression model for throughput prediction
 1 file changed, 1 insertion(+)
 create mode 100644 models/xgb_model.json
/bin/bash: line 1: YOUR_TOKEN: No such file or directory
